In [32]:
import pandas as pd
df = pd.read_excel("frame_gold.xlsx", sheet_name=0)

df.head()



,uuid,title,protest_excerpt,title_frame,excerpt_frame,demands_present,authority_quote,protester_quote,thematic_present
0,d1334539-4cf3-4c5f-a0ef-88db5f167395,Third politician in a month killed in Ecuador,The gang boss was moved out of a jail with lig...,residual,grievance,yes,no,no,no
1,b088e93f-1de4-4021-97bf-a90700b0a0ac,Amazon CEO tells staff to work in office three...,This comes after Amazon said it would lay off ...,residual,widespread_support,yes,no,yes,yes
2,fffb4a3a-8ee2-4cc4-9b9a-b0f676c5bdad,Gaza ceasefire protest in Capitol rotunda lead...,Capitol rotunda on Sept. 27. Photo: Michael Ro...,grievance,disruption,yes,yes,no,yes
3,155faae5-c8b6-441e-a20e-dce06e01cf32,"Daniel Penny, accused in Jordan Neely subway c...",Neely and could not have foreseen his untimely...,residual,grievance,yes,yes,yes,no
4,50fea669-6828-44de-8a70-b448e398be11,Legal scholars raising constitutional argument...,Not all in the legal community agree – and wha...,residual,protester_violence,no,yes,no,yes


In [19]:
df['excerpt_frame'].value_counts()

excerpt_frame
protester_violence    31
grievance             25
disruption            17
widespread_support    14
harm_to_protesters    12
impactful              8
residual               6
property_damage        4
peaceful               3
Name: count, dtype: int64

# Imports, prompts, load data

In [36]:
# If needed, run once in your environment:
# %pip install openai pandas openpyxl scikit-learn simpledorff

import os
import json
import time
import pandas as pd
import numpy as np

from openai import OpenAI
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    classification_report,
    confusion_matrix
)
import simpledorff

# --------------------------------------------------
# IMPORT ONE PROMPT
# --------------------------------------------------

# To switch prompts, change ONLY this import.
import frame_prompts
importlib.reload(frame_prompts)

from frame_prompts import OPERATIONALIZED_JOINT_PROMPT as SYSTEM_PROMPT

# Example alternative:
# from frame_prompts import OPERATIONALIZED_JOINT_PROMPT as SYSTEM_PROMPT


# --------------------------------------------------
# MODEL SETTINGS
# --------------------------------------------------

MODEL = "gpt-5"
REASONING_EFFORT = "medium"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


# --------------------------------------------------
# LOAD GOLD DATA
# --------------------------------------------------

GOLD_FILE = "frame_gold.xlsx"

df = pd.read_excel(GOLD_FILE)

required_columns = [
    "uuid",
    "title",
    "protest_excerpt",
    "title_frame",
    "excerpt_frame",
    "demands_present",
    "authority_quote",
    "protester_quote",
    "thematic_present",
]

missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df[required_columns].copy()

# Normalize gold labels for comparison with GPT output.
label_columns = [
    "title_frame",
    "excerpt_frame",
    "demands_present",
    "authority_quote",
    "protester_quote",
    "thematic_present",
]

for col in label_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

# Make sure input text is not NaN.
df["title"] = df["title"].fillna("").astype(str)
df["protest_excerpt"] = df["protest_excerpt"].fillna("").astype(str)

print(f"Loaded {len(df)} gold-standard samples.")
print(df[label_columns].head())

test_df = df.head(3)
print(test_df[label_columns].head())

Loaded 120 gold-standard samples.
  title_frame       excerpt_frame demands_present authority_quote  \
0    residual           grievance             yes              no   
1    residual  widespread_support             yes              no   
2   grievance          disruption             yes             yes   
3    residual           grievance             yes             yes   
4    residual  protester_violence              no             yes   

  protester_quote thematic_present  
0              no               no  
1             yes              yes  
2              no              yes  
3             yes               no  
4              no              yes  
  title_frame       excerpt_frame demands_present authority_quote  \
0    residual           grievance             yes              no   
1    residual  widespread_support             yes              no   
2   grievance          disruption             yes             yes   

  protester_quote thematic_present  
0              

# API call

In [ ]:
print(SYSTEM_PROMPT)

In [39]:
# --------------------------------------------------
# STRICT OUTPUT SCHEMA
# --------------------------------------------------

FRAME_LABELS = [
    "protester_violence",
    "harm_to_protesters",
    "property_damage",
    "disruption",
    "grievance",
    "widespread_support",
    "peaceful",
    "impactful",
    "residual",
]

ANNOTATION_SCHEMA = {
    "type": "object",
    "properties": {
        "title_frame": {
            "type": "string",
            "enum": FRAME_LABELS,
        },
        "excerpt_frame": {
            "type": "string",
            "enum": FRAME_LABELS,
        },
        "demands_present": {
            "type": "string",
            "enum": ["yes", "no"],
        },
        "protester_quote": {
            "type": "string",
            "enum": ["yes", "no"],
        },
        "authority_quote": {
            "type": "string",
            "enum": ["yes", "no"],
        },
        "thematic_present": {
            "type": "string",
            "enum": ["yes", "no"],
        },
    },
    "required": [
        "title_frame",
        "excerpt_frame",
        "demands_present",
        "protester_quote",
        "authority_quote",
        "thematic_present",
    ],
    "additionalProperties": False,
}


# --------------------------------------------------
# FORMAT EACH SAMPLE
# --------------------------------------------------

def format_input(title, excerpt):
    return f"""TITLE:
{title}

EXCERPT:
{excerpt}
"""


# --------------------------------------------------
# CALL GPT
# --------------------------------------------------

def annotate_sample(title, excerpt, max_retries=3):

    user_input = format_input(title, excerpt)

    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=MODEL,

                # Stable coding instructions
                instructions=SYSTEM_PROMPT,

                # Only TITLE + EXCERPT change between cases
                input=user_input,

                reasoning={
                    "effort": REASONING_EFFORT
                },

                text={
                    "verbosity": "low",
                    "format": {
                        "type": "json_schema",
                        "name": "protest_frame_annotation",
                        "strict": True,
                        "schema": ANNOTATION_SCHEMA,
                    },
                },
            )

            annotation = json.loads(response.output_text)

            return {
                **annotation,
                "response_id": response.id,
                "input_tokens": response.usage.input_tokens,
                "output_tokens": response.usage.output_tokens,
                "error": None,
            }

        except Exception as e:
            print(
                f"Attempt {attempt + 1}/{max_retries} failed: "
                f"{type(e).__name__}: {e}"
            )

            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                return {
                    "title_frame": None,
                    "excerpt_frame": None,
                    "demands_present": None,
                    "protester_quote": None,
                    "authority_quote": None,
                    "thematic_present": None,
                    "response_id": None,
                    "input_tokens": None,
                    "output_tokens": None,
                    "error": str(e),
                }


# --------------------------------------------------
# RUN VALIDATION SAMPLE
# --------------------------------------------------

results = []

for i, row in df.iterrows():

    print(f"[{i + 1}/{len(df)}] {row['uuid']}")

    prediction = annotate_sample(
        title=row["title"],
        excerpt=row["protest_excerpt"],
    )

    results.append(prediction)


pred_df = pd.DataFrame(results)

# Add clear GPT prefixes.
pred_df = pred_df.rename(columns={
    "title_frame": "gpt_title_frame",
    "excerpt_frame": "gpt_excerpt_frame",
    "demands_present": "gpt_demands_present",
    "protester_quote": "gpt_protester_quote",
    "authority_quote": "gpt_authority_quote",
    "thematic_present": "gpt_thematic_present",
})

results_df = pd.concat(
    [
        df.reset_index(drop=True),
        pred_df.reset_index(drop=True)
    ],
    axis=1
)

print("\nFinished.")
print(f"Successful: {results_df['error'].isna().sum()}")
print(f"Errors:     {results_df['error'].notna().sum()}")

print(
    "\nTotal input tokens:",
    results_df["input_tokens"].fillna(0).sum()
)

print(
    "Total output tokens:",
    results_df["output_tokens"].fillna(0).sum()
)

# SAVE IMMEDIATELY
PROMPT_NAME = "operationalized_joint"
output_file = f"frame_validation_{PROMPT_NAME}_{MODEL}.xlsx"
results_df.to_excel(output_file, index=False)

print(f"Raw predictions saved to: {output_file}")

results_df.head()

[1/120] d1334539-4cf3-4c5f-a0ef-88db5f167395
[2/120] b088e93f-1de4-4021-97bf-a90700b0a0ac
[3/120] fffb4a3a-8ee2-4cc4-9b9a-b0f676c5bdad
[4/120] 155faae5-c8b6-441e-a20e-dce06e01cf32
[5/120] 50fea669-6828-44de-8a70-b448e398be11
[6/120] 81efb265-e3ba-4508-adc7-73cbc029f359
[7/120] 5fb219a8-9df7-4ad0-bc57-d46be3e327e0
[8/120] 252fb33d-5e6c-4cff-acc0-98c253f6c4b8
[9/120] 4acf5304-e74d-4a4f-bee0-caf65bd02abe
[10/120] 74fd09e2-35a3-495b-9b59-1610ec3398e1
[11/120] 7d9d161f-8087-4e0b-89e2-4a9bc1dcfd9d
[12/120] 0c5fa89f-f32f-4014-98e5-e8ad481b8a2f
[13/120] b2a2ef91-319a-47bf-9a61-1f8b4e192abf
[14/120] a92ef5b3-9891-4492-8a32-5024d7cc52dc
[15/120] c13350d6-ae7e-43f9-8999-17e530a4c80b
[16/120] 874c16ec-4684-40c7-8928-06baaa20260d
[17/120] 0192cd22-a165-4eb8-bf6d-da6165164b2e
[18/120] 95c2c809-f17e-4d9d-af74-26ba987929cd
[19/120] 109cd735-179b-4d53-ba32-ac1b70d5d990
[20/120] 36ab3319-2965-4075-80bd-14627327ab65
[21/120] f4bf7615-7d65-4e3d-b332-fbf590087f99
[22/120] 58a57f20-02b7-4389-9add-4cd5c76aae

,uuid,title,protest_excerpt,title_frame,excerpt_frame,demands_present,authority_quote,protester_quote,thematic_present,gpt_title_frame,gpt_excerpt_frame,gpt_demands_present,gpt_protester_quote,gpt_authority_quote,gpt_thematic_present,response_id,input_tokens,output_tokens,error
0,d1334539-4cf3-4c5f-a0ef-88db5f167395,Third politician in a month killed in Ecuador,The gang boss was moved out of a jail with lig...,residual,grievance,yes,no,no,no,residual,grievance,yes,no,no,no,resp_01105adc0d5748b9006a84608c036481a09400f8c...,4448,600,None
1,b088e93f-1de4-4021-97bf-a90700b0a0ac,Amazon CEO tells staff to work in office three...,This comes after Amazon said it would lay off ...,residual,widespread_support,yes,no,yes,yes,residual,grievance,yes,yes,yes,no,resp_03eb96477bdaf400006a846094ae7087d2a3471e4...,4514,830,None
2,fffb4a3a-8ee2-4cc4-9b9a-b0f676c5bdad,Gaza ceasefire protest in Capitol rotunda lead...,Capitol rotunda on Sept. 27. Photo: Michael Ro...,grievance,disruption,yes,yes,no,yes,residual,disruption,yes,no,yes,yes,resp_00df05148f0c6ee0006a8460a0cf1487d29ca229b...,4506,1046,None
3,155faae5-c8b6-441e-a20e-dce06e01cf32,"Daniel Penny, accused in Jordan Neely subway c...",Neely and could not have foreseen his untimely...,residual,grievance,yes,yes,yes,no,residual,grievance,yes,no,yes,no,resp_0d3c76e83129a419006a8460c0a65487d2997d30a...,4450,629,None
4,50fea669-6828-44de-8a70-b448e398be11,Legal scholars raising constitutional argument...,Not all in the legal community agree – and wha...,residual,protester_violence,no,yes,no,yes,residual,residual,no,no,yes,yes,resp_08e6e65d877ec641006a8460ce811c81a2b2da562...,4748,1210,None


# results and analysis

In [40]:
# --------------------------------------------------
# EVALUATION FUNCTIONS
# --------------------------------------------------

TASKS = {
    "title_frame": {
        "gold": "title_frame",
        "pred": "gpt_title_frame",
        "type": "multiclass",
    },
    "excerpt_frame": {
        "gold": "excerpt_frame",
        "pred": "gpt_excerpt_frame",
        "type": "multiclass",
    },
    "demands_present": {
        "gold": "demands_present",
        "pred": "gpt_demands_present",
        "type": "binary",
    },
    "protester_quote": {
        "gold": "protester_quote",
        "pred": "gpt_protester_quote",
        "type": "binary",
    },
    "authority_quote": {
        "gold": "authority_quote",
        "pred": "gpt_authority_quote",
        "type": "binary",
    },
    "thematic_present": {
        "gold": "thematic_present",
        "pred": "gpt_thematic_present",
        "type": "binary",
    },
}


def kripp_alpha_two_annotators(gold, pred):
    """
    Krippendorff's alpha for nominal labels,
    treating Human Gold and GPT as two annotators.
    """

    tmp = pd.DataFrame({
        "item_id": np.arange(len(gold)).repeat(2),
        "annotator_id": ["human", "gpt"] * len(gold),
        "label": np.column_stack([gold, pred]).reshape(-1),
    })

    return simpledorff.calculate_krippendorffs_alpha_for_df(
        tmp,
        experiment_col="item_id",
        annotator_col="annotator_id",
        class_col="label",
    )


def evaluate_task(data, gold_col, pred_col, task_type):

    valid = data[[gold_col, pred_col]].dropna()

    y_true = valid[gold_col].astype(str)
    y_pred = valid[pred_col].astype(str)

    result = {
        "n": len(valid),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "kripp_alpha": kripp_alpha_two_annotators(
            y_true.tolist(),
            y_pred.tolist(),
        ),
    }

    if task_type == "binary":

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=["yes"],
            average=None,
            zero_division=0,
        )

        result["yes_precision"] = precision[0]
        result["yes_recall"] = recall[0]
        result["yes_f1"] = f1[0]

    else:

        result["weighted_f1"] = f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        )

    return result


# --------------------------------------------------
# SUMMARY TABLE
# --------------------------------------------------

summary_rows = []

for task_name, config in TASKS.items():

    metrics = evaluate_task(
        results_df,
        gold_col=config["gold"],
        pred_col=config["pred"],
        task_type=config["type"],
    )

    summary_rows.append({
        "task": task_name,
        **metrics,
    })


summary_df = pd.DataFrame(summary_rows)

# Arrange the most useful columns first.
preferred_cols = [
    "task",
    "n",
    "accuracy",
    "macro_f1",
    "yes_precision",
    "yes_recall",
    "yes_f1",
    "weighted_f1",
    "kripp_alpha",
]

summary_df = summary_df[
    [c for c in preferred_cols if c in summary_df.columns]
]

print("\n==============================")
print("VALIDATION SUMMARY")
print("==============================\n")

display(
    summary_df.round(3)
)


# --------------------------------------------------
# DETAILED MULTICLASS REPORTS
# --------------------------------------------------

for task in ["title_frame", "excerpt_frame"]:

    gold_col = TASKS[task]["gold"]
    pred_col = TASKS[task]["pred"]

    valid = results_df[[gold_col, pred_col]].dropna()

    print("\n")
    print("=" * 70)
    print(task.upper())
    print("=" * 70)

    print(
        classification_report(
            valid[gold_col],
            valid[pred_col],
            labels=FRAME_LABELS,
            zero_division=0,
        )
    )

    cm = pd.DataFrame(
        confusion_matrix(
            valid[gold_col],
            valid[pred_col],
            labels=FRAME_LABELS,
        ),
        index=[f"gold_{x}" for x in FRAME_LABELS],
        columns=[f"gpt_{x}" for x in FRAME_LABELS],
    )

    display(cm)


# --------------------------------------------------
# SAVE summary
# --------------------------------------------------
summary_df.to_excel(
    f"frame_validation_summary_{PROMPT_NAME}_{MODEL}.xlsx",
    index=False
)


VALIDATION SUMMARY



,task,n,accuracy,macro_f1,yes_precision,yes_recall,yes_f1,weighted_f1,kripp_alpha
0,title_frame,120,0.775,0.594,NaN,NaN,NaN,0.774,0.588
1,excerpt_frame,120,0.600,0.579,NaN,NaN,NaN,0.605,0.528
2,demands_present,120,0.842,0.803,0.856,0.928,0.890,NaN,0.608
3,protester_quote,120,0.875,0.870,0.891,0.804,0.845,NaN,0.742
4,authority_quote,120,0.750,0.680,0.723,0.973,0.830,NaN,0.363
5,thematic_present,120,0.783,0.778,0.760,0.731,0.745,NaN,0.559




TITLE_FRAME
                    precision    recall  f1-score   support

protester_violence       0.40      0.40      0.40         5
harm_to_protesters       0.57      0.67      0.62         6
   property_damage       0.50      1.00      0.67         1
        disruption       0.25      0.25      0.25         4
         grievance       0.69      0.64      0.67        14
widespread_support       0.83      0.83      0.83         6
          peaceful       0.00      0.00      0.00         0
         impactful       0.50      0.40      0.44         5
          residual       0.87      0.87      0.87        79

          accuracy                           0.78       120
         macro avg       0.51      0.56      0.53       120
      weighted avg       0.78      0.78      0.77       120



,gpt_protester_violence,gpt_harm_to_protesters,gpt_property_damage,gpt_disruption,gpt_grievance,gpt_widespread_support,gpt_peaceful,gpt_impactful,gpt_residual
gold_protester_violence,2,0,1,0,0,0,0,0,2
gold_harm_to_protesters,0,4,0,0,0,0,0,0,2
gold_property_damage,0,0,1,0,0,0,0,0,0
gold_disruption,0,0,0,1,0,0,0,1,2
gold_grievance,0,0,0,1,9,0,0,0,4
gold_widespread_support,0,0,0,0,1,5,0,0,0
gold_peaceful,0,0,0,0,0,0,0,0,0
gold_impactful,1,0,0,2,0,0,0,2,0
gold_residual,2,3,0,0,3,1,0,1,69




EXCERPT_FRAME
                    precision    recall  f1-score   support

protester_violence       0.73      0.61      0.67        31
harm_to_protesters       0.69      0.92      0.79        12
   property_damage       0.75      0.75      0.75         4
        disruption       0.45      0.29      0.36        17
         grievance       0.64      0.72      0.68        25
widespread_support       0.88      0.50      0.64        14
          peaceful       1.00      0.33      0.50         3
         impactful       0.50      0.50      0.50         8
          residual       0.22      0.67      0.33         6

          accuracy                           0.60       120
         macro avg       0.65      0.59      0.58       120
      weighted avg       0.65      0.60      0.60       120



,gpt_protester_violence,gpt_harm_to_protesters,gpt_property_damage,gpt_disruption,gpt_grievance,gpt_widespread_support,gpt_peaceful,gpt_impactful,gpt_residual
gold_protester_violence,19,2,0,2,1,0,0,2,5
gold_harm_to_protesters,1,11,0,0,0,0,0,0,0
gold_property_damage,1,0,3,0,0,0,0,0,0
gold_disruption,5,0,0,5,2,0,0,2,3
gold_grievance,0,1,0,1,18,1,0,0,4
gold_widespread_support,0,0,1,1,4,7,0,0,1
gold_peaceful,0,1,0,0,1,0,1,0,0
gold_impactful,0,0,0,2,1,0,0,4,1
gold_residual,0,1,0,0,1,0,0,0,4


In [41]:
# ============================================================
# THEMATIC_PRESENT DEEP DIVE
# ============================================================

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

gold_col = "thematic_present"
pred_col = "gpt_thematic_present"

thematic_df = results_df[
    [
        "uuid",
        "title",
        "protest_excerpt",
        gold_col,
        pred_col
    ]
].dropna().copy()


# ------------------------------------------------------------
# 1. GOLD VS GPT PREVALENCE
# ------------------------------------------------------------

print("=" * 60)
print("THEMATIC PRESENT: PREVALENCE")
print("=" * 60)

gold_counts = thematic_df[gold_col].value_counts()
pred_counts = thematic_df[pred_col].value_counts()

gold_yes_rate = (thematic_df[gold_col] == "yes").mean()
pred_yes_rate = (thematic_df[pred_col] == "yes").mean()

print("\nGold labels:")
print(gold_counts)

print("\nGPT labels:")
print(pred_counts)

print(f"\nGold YES rate: {gold_yes_rate:.3f}")
print(f"GPT YES rate:  {pred_yes_rate:.3f}")
print(f"Difference:    {pred_yes_rate - gold_yes_rate:+.3f}")


# ------------------------------------------------------------
# 2. CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    thematic_df[gold_col],
    thematic_df[pred_col],
    labels=["no", "yes"]
)

cm_df = pd.DataFrame(
    cm,
    index=["Gold NO", "Gold YES"],
    columns=["GPT NO", "GPT YES"]
)

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

display(cm_df)

tn, fp, fn, tp = cm.ravel()

print(f"\nTrue negatives:  {tn}")
print(f"False positives: {fp}")
print(f"False negatives: {fn}")
print(f"True positives:  {tp}")


# ------------------------------------------------------------
# 3. METRICS FOR BOTH YES AND NO
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        thematic_df[gold_col],
        thematic_df[pred_col],
        labels=["no", "yes"],
        digits=3,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 4. FALSE POSITIVES
# Gold = NO, GPT = YES
# ------------------------------------------------------------

false_positives = thematic_df[
    (thematic_df[gold_col] == "no") &
    (thematic_df[pred_col] == "yes")
].copy()

print("\n" + "=" * 60)
print(f"FALSE POSITIVES: {len(false_positives)}")
print("=" * 60)

display(
    false_positives[
        ["uuid", "title", "protest_excerpt"]
    ].reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. FALSE NEGATIVES
# Gold = YES, GPT = NO
# ------------------------------------------------------------

false_negatives = thematic_df[
    (thematic_df[gold_col] == "yes") &
    (thematic_df[pred_col] == "no")
].copy()

print("\n" + "=" * 60)
print(f"FALSE NEGATIVES: {len(false_negatives)}")
print("=" * 60)

display(
    false_negatives[
        ["uuid", "title", "protest_excerpt"]
    ].reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. OPTIONAL: SAVE ERROR CASES FOR MANUAL REVIEW
# ------------------------------------------------------------

false_positives.to_excel(
    "thematic_false_positives.xlsx",
    index=False
)

false_negatives.to_excel(
    "thematic_false_negatives.xlsx",
    index=False
)

print("\nSaved thematic false positives and false negatives.")

THEMATIC PRESENT: PREVALENCE

Gold labels:
thematic_present
no     68
yes    52
Name: count, dtype: int64

GPT labels:
gpt_thematic_present
no     70
yes    50
Name: count, dtype: int64

Gold YES rate: 0.433
GPT YES rate:  0.417
Difference:    -0.017

CONFUSION MATRIX


,GPT NO,GPT YES
Gold NO,56,12
Gold YES,14,38



True negatives:  56
False positives: 12
False negatives: 14
True positives:  38

CLASSIFICATION REPORT
              precision    recall  f1-score   support

          no      0.800     0.824     0.812        68
         yes      0.760     0.731     0.745        52

    accuracy                          0.783       120
   macro avg      0.780     0.777     0.778       120
weighted avg      0.783     0.783     0.783       120


FALSE POSITIVES: 12


,uuid,title,protest_excerpt
0,0192cd22-a165-4eb8-bf6d-da6165164b2e,Woman who threatened to hang Pelosi during Jan...,Rioters attempting to breach the U.S. Capitol ...
1,c388a480-bee6-4308-8844-c8bb0b12c7a0,Russian court sets date for new trial of jaile...,The Moscow City Court held a preliminary heari...
2,5cc26d4f-b198-4dbb-a8a9-7ab848a1f997,Australia news live: police search Sydney Harb...,7m ago 19.20 EDT Rallies against 10 years of o...
3,50275382-1755-4359-bcfb-baccd1b2e65e,"HBO drops ""The Wire"" creator David Simon amid ...","President Biden calls for ""fair deal"" for writ..."
4,1c878f00-1989-464c-bc13-e00e0dba2e68,"2 Years After Condemning Violence, Trump Now F...","6, it was the largest crowd I have ever spoken..."
5,2d6f8144-9af6-4010-a23c-c918d4ad27f2,Auto Workers Strike At Stellantis’ Ram Pickup ...,LOADING ERROR LOADING\nWorkers went on strike ...
6,67f71419-adca-496c-b73a-dde4db7f3191,Idaho man who dangled from Senate balcony duri...,An Idaho man who traveled to Washington in a c...
7,83c2698d-75ca-47d9-9d2f-1dd18c897dd2,Israelis stage massive protests after governme...,Why Israel's judicial overhaul is causing such...
8,832062d2-1112-41f4-8409-64e22ed48541,Supreme Court Skeptical of Biden Admin in Jan ...,Solicitor General Elizabeth Prelogar argued Tu...
9,e26eecb3-6e5f-456d-b05a-fc7008ed9de9,Trump claims he can't get a fair trial in DC a...,"""No way I can get a fair trial, or even close ..."



FALSE NEGATIVES: 14


,uuid,title,protest_excerpt
0,b088e93f-1de4-4021-97bf-a90700b0a0ac,Amazon CEO tells staff to work in office three...,This comes after Amazon said it would lay off ...
1,0c5fa89f-f32f-4014-98e5-e8ad481b8a2f,House censures Rep. Rashida Tlaib amid biparti...,Four members voted present. Tlaib has been one...
2,c13350d6-ae7e-43f9-8999-17e530a4c80b,"100 days into Israel-Hamas war, piano installe...",Piano at Washington Square Park a symbol of un...
3,6970867e-b4b2-4487-b134-da9903b374b6,Columbia University protests continue for 5th ...,Columbia University protests expected for 5th ...
4,a09f3465-a310-4716-882a-10a9ac5a6e16,The judge assigned to Trump’s Jan. 6 case is a...,WASHINGTON (AP) — The federal judge assigned t...
5,2da731fa-6040-4412-85a3-5ab7415ba5ff,Greta Thunberg charged with disobeying police ...,Swedish climate activist Greta Thunberg has be...
6,3001dd1e-4ea2-4e94-83ad-d9bbcc5275c8,Fox News Guest Suggests Using ‘Hillary’s Depro...,A guest on Fox News said Wednesday that colleg...
7,df34b27f-dd95-4afe-9806-1511dbccbc26,Hezbollah terror flag found at Princeton's ant...,Students at elite college campuses across the ...
8,9646b88d-3a40-4bef-8f10-3ba21a8df555,Demonstrators in Chicago Loop rally for ceasef...,Demonstrators rally in support of Palestine in...
9,0bda2100-bb28-4934-a24a-bc28ea037572,Anti-Israel agitators shut down Senate cafeter...,More than 50 people were arrested Tuesday afte...



Saved thematic false positives and false negatives.


In [42]:
# ============================================
# ESTIMATE 9K PRODUCTION COST FROM 120 RUN
# ============================================

N_PRODUCTION = 9000

avg_input = results_df["input_tokens"].mean()
avg_output = results_df["output_tokens"].mean()

total_input_9k = avg_input * N_PRODUCTION
total_output_9k = avg_output * N_PRODUCTION

print(f"Average input tokens/article:  {avg_input:,.0f}")
print(f"Average output tokens/article: {avg_output:,.0f}")

print(f"\nProjected input tokens for 9k:  {total_input_9k:,.0f}")
print(f"Projected output tokens for 9k: {total_output_9k:,.0f}")

Average input tokens/article:  4,699
Average output tokens/article: 1,298

Projected input tokens for 9k:  42,290,400
Projected output tokens for 9k: 11,679,450


In [44]:
# ============================================
# ESTIMATE GPT-5 PRODUCTION COST
# ============================================

N_ARTICLES = 9000

# Observed averages from validation run
AVG_INPUT_TOKENS = 4699
AVG_OUTPUT_TOKENS = 1298

# GPT-5 standard pricing per 1M tokens
INPUT_PRICE = 1.25
OUTPUT_PRICE = 10.00


# Project token usage
total_input_tokens = AVG_INPUT_TOKENS * N_ARTICLES
total_output_tokens = AVG_OUTPUT_TOKENS * N_ARTICLES

# Costs
input_cost = (total_input_tokens / 1_000_000) * INPUT_PRICE
output_cost = (total_output_tokens / 1_000_000) * OUTPUT_PRICE
total_cost = input_cost + output_cost


print(f"Articles: {N_ARTICLES:,}")

print(f"\nProjected input tokens:  {total_input_tokens:,.0f}")
print(f"Projected output tokens: {total_output_tokens:,.0f}")

print(f"\nInput cost:  ${input_cost:,.2f}")
print(f"Output cost: ${output_cost:,.2f}")
print(f"TOTAL COST:  ${total_cost:,.2f}")

Articles: 9,000

Projected input tokens:  42,291,000
Projected output tokens: 11,682,000

Input cost:  $52.86
Output cost: $116.82
TOTAL COST:  $169.68
